# Lecția 8 · Evaluare și validare

Aici se câștigă sau se pierde concursul. Un model care pare bun pe clasamentul public poate fi slab pe cel ascuns, care contează la final. Modulul ăsta e despre cum să ai încredere justificată în scorul tău. E cea mai puțin spectaculoasă parte și cea care face diferența dintre un top 10 și un mijloc de clasament.

*Celulele de mai jos sunt codul din lecție, în ordine. Unele presupun date pe care le încarci tu (`df`, `X_train`, `y_train`).*

## 01 · De ce acuratețea te minte

Acuratețea e procentul de predicții corecte. Sună rezonabil, dar pe clase dezechilibrate e înșelătoare. Dacă 98% din exemple sunt clasa nu, un model care spune mereu nu are 98% acuratețe și zero valoare. De asta ai nevoie de metrici care se uită la fiecare clasă separat.

## 02 · Matricea de confuzie: temelia

Matricea de confuzie numără cele patru feluri de rezultat pentru o clasificare binară. E baza din care se calculează toate celelalte metrici, deci merită înțeleasă bine.

- TP (adevărat pozitiv): era pozitiv, ai zis pozitiv. Corect.
- TN (adevărat negativ): era negativ, ai zis negativ. Corect.
- FP (fals pozitiv): era negativ, ai zis pozitiv. Alarmă falsă.
- FN (fals negativ): era pozitiv, ai zis negativ. L-ai ratat.

Ce te doare mai tare depinde de problemă. La un test de boală, un fals negativ (bolnav trimis acasă) e mai grav decât un fals pozitiv. La un filtru de spam, invers. Metrica bună reflectă ce cost real are fiecare tip de greșeală.

## 03 · Precizie, recall, F1

**precizie = TP / (TP + FP)**

Din câte ai zis că sunt pozitive, câte chiar erau. Precizie mare = puține alarme false.

**recall = TP / (TP + FN)**

Din câte erau de fapt pozitive, câte ai prins. Recall mare = ratezi puține.

Precizia și recall-ul trag în direcții opuse. Dacă spui pozitiv doar când ești foarte sigur, precizia crește dar recall-ul scade. Dacă spui pozitiv des, invers. F1 le împacă într-un singur număr, media lor armonică, care e mică dacă oricare din ele e mică.

**F1 = 2 · (precizie · recall) / (precizie + recall)**

Media armonică. Pedepsește dezechilibrul: nu poți avea F1 mare cu recall minuscul.

> **Reține:** Când ai multe clase, F1 se agregă. Macro-F1 face media pe clase tratându-le egal (bun când îți pasă de clasele rare). Weighted-F1 ține cont de mărimea fiecărei clase. Citește în regulament care se punctează și optimizează exact pe aia.

## 04 · Validare fără scurgeri

O scurgere de informație (data leakage) e când, fără să vrei, informație din test ajunge în antrenare. Rezultatul: scor local grozav care se prăbușește pe clasamentul real. E cel mai perfid mod de a pierde puncte, pentru că totul pare în regulă.

Aperi-te împărțind datele corect. Ține un set de validare pe care modelul nu-l vede la antrenare și pe care îl folosești ca să estimezi scorul real. Mai bine, folosește k-fold: împarți datele în k părți, antrenezi pe k-1 și testezi pe una, rotind, apoi mediezi. Așa folosești toate datele și ai o estimare mai stabilă.

Când clasele sunt dezechilibrate, folosește k-fold stratificat, care păstrează proporția claselor în fiecare fold. Altfel un fold poate să nu conțină deloc clasa rară. Când datele au timp sau grupuri (același pacient în mai multe rânduri), split-ul trebuie să le respecte, altfel modelul trage cu ochiul.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
scoruri = cross_val_score(model, X, y, cv=cv, scoring="f1_macro")
print(scoruri.mean(), scoruri.std())

## 05 · Bias, varianță și curba de învățare

Două boli opuse. Bias mare (underfitting): modelul e prea simplu, merge slab și pe antrenare, și pe validare. Varianță mare (overfitting): modelul e prea complex, merge grozav pe antrenare dar slab pe validare, pentru că a memorat.

Curba de învățare le diagnostichează: desenezi scorul de antrenare și cel de validare pe măsură ce crești datele. Dacă amândouă sunt joase și apropiate, ai bias, îți trebuie model mai puternic. Dacă e o prăpastie mare între ele, ai varianță, îți trebuie mai multe date sau regularizare.

## 06 · Cele două submisii finale

> **Reține:** Regula de aur: la final alege conștient cele două submisii. Una pe scorul tău local (validarea) cel mai bun, una pe clasamentul public cel mai bun. Dacă local și public sunt de acord, ai încredere. Dacă diferă mult, ai o scurgere sau un split prost. Nu lăsa submisiile finale pe ultimele trimise din reflex.